# NB4 — Medallion Pipeline (Bronze → Silver → Gold), lightweight

**Use case:** LLM observability — exact schema from slide §8 (Lakehouse cho AI/ML) medallion frame.
Maps to deliverable bullet 4 (the Milestone-1 Lakehouse artifact).

Pre-req: ran `make data` — but if you jumped straight here, the cell below
generates the Bronze sample for you rather than failing on a missing path.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
from pathlib import Path

import polars as pl
import duckdb
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

BRONZE = path("bronze", "llm_calls_raw")
SILVER = path("silver", "llm_calls")
GOLD   = path("gold",   "llm_daily_metrics")

# Self-healing pre-req (same pattern as NB7/NB8). Without this, skipping
# `make data` surfaces as a raw `Os { code: 2, kind: NotFound }` from the Rust
# layer — technically correct, useless to a student.
if not Path(BRONZE).exists():
    print("Bronze not found — running scripts/generate_data_lite.py first ...")
    import generate_data_lite

    generate_data_lite.main()

## Bronze — verify raw is loaded

In [2]:
bronze_n = DeltaTable(BRONZE).to_pyarrow_table().num_rows
print(f"Bronze rows: {bronze_n:,}")
print(pl.from_arrow(DeltaTable(BRONZE).to_pyarrow_table().slice(0, 2)))

Bronze rows: 200,000
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬─────────────────────────────────┐
│ request_id                      ┆ ts                      ┆ raw_json                        │
│ ---                             ┆ ---                     ┆ ---                             │
│ str                             ┆ datetime[μs, UTC]       ┆ str                             │
╞═════════════════════════════════╪═════════════════════════╪═════════════════════════════════╡
│ 43151841-daf7-4447-9a7c-3263ce… ┆ 2026-04-01 00:00:00 UTC ┆ {"model": "claude-sonnet-4-6",… │
│ d503348f-0eeb-4dc6-88fd-301e66… ┆ 2026-04-01 00:00:03 UTC ┆ {"model": "claude-haiku-4-5", … │
└─────────────────────────────────┴─────────────────────────┴─────────────────────────────────┘


## Silver — parse, validate, dedup

Rules: drop malformed JSON, dedupe by `request_id`, project typed columns.

In [3]:
reset(SILVER)

# DuckDB does the JSON parse + dedup in one query — Polars also works,
# DuckDB just has nicer JSON syntax for this case.
# DuckDB reads Delta through Arrow, not through `delta_scan()`. The latter
# autoloads an extension over the network; Arrow registration is offline and
# zero-copy, so the lab works on a locked-down machine.
con = duckdb.connect()
con.register("bronze", DeltaTable(BRONZE).to_pyarrow_table())

silver_arrow = con.sql(f"""
    WITH parsed AS (
      SELECT
        request_id,
        ts,
        CAST(ts AS DATE)                            AS date,
        json_extract_string(raw_json, '$.model')          AS model,
        json_extract_string(raw_json, '$.user_id')        AS user_id,
        CAST(json_extract(raw_json, '$.usage.input')  AS INTEGER) AS prompt_tokens,
        CAST(json_extract(raw_json, '$.usage.output') AS INTEGER) AS completion_tokens,
        CAST(json_extract(raw_json, '$.latency_ms')   AS INTEGER) AS latency_ms,
        json_extract_string(raw_json, '$.status')         AS status,
        ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts) AS rn
      FROM bronze
    )
    SELECT request_id, ts, date, model, user_id,
           prompt_tokens, completion_tokens, latency_ms, status
    FROM parsed
    WHERE rn = 1 AND model IS NOT NULL
""").arrow()

write_deltalake(SILVER, silver_arrow, mode="overwrite", partition_by=["date"])

silver_n = DeltaTable(SILVER).to_pyarrow_table().num_rows
print(f"Silver rows: {silver_n:,}  (Bronze {bronze_n:,} → dedup dropped {bronze_n - silver_n:,})")
assert silver_n < bronze_n, (
    "Silver has the same row count as Bronze — dedup did not run. "
    "Did you regenerate Bronze with the latest generator (which injects retries)?"
)

Silver rows: 190,052  (Bronze 200,000 → dedup dropped 9,948)


## Gold — aggregate to (date, model) metrics

In [4]:
reset(GOLD)

# Illustrative cost model — NOT canonical pricing.
# (input USD / 1M tokens, output USD / 1M tokens)
COST_TABLE = """
  VALUES
    ('claude-haiku-4-5',  0.80,  4.00),
    ('claude-sonnet-4-6', 3.00, 15.00),
    ('claude-opus-4-7', 15.00, 75.00)
"""

con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
gold_arrow = con.sql(f"""
    WITH cost(model, c_in, c_out) AS ({COST_TABLE})
    SELECT
      s.date,
      s.model,
      QUANTILE_CONT(s.latency_ms, 0.50) AS p50_latency_ms,
      QUANTILE_CONT(s.latency_ms, 0.95) AS p95_latency_ms,
      SUM(s.prompt_tokens)              AS total_prompt_tokens,
      SUM(s.completion_tokens)          AS total_completion_tokens,
      AVG(CASE WHEN s.status <> 'ok' THEN 1.0 ELSE 0.0 END) AS error_rate,
      (SUM(s.prompt_tokens)     * c.c_in  / 1e6) +
      (SUM(s.completion_tokens) * c.c_out / 1e6) AS cost_usd
    FROM silver s
    JOIN cost c USING (model)
    GROUP BY s.date, s.model, c.c_in, c.c_out
    ORDER BY s.date, s.model
""").arrow()

write_deltalake(GOLD, gold_arrow, mode="overwrite", partition_by=["date"])

# Z-order for fast filter-by-model dashboards
DeltaTable(GOLD).optimize.z_order(["model"])

{'numFilesAdded': 8,
 'numFilesRemoved': 8,
 'filesAdded': '{"avg":2768.75,"max":2770,"min":2766,"totalFiles":8,"totalSize":22150}',
 'filesRemoved': '{"avg":2687.875,"max":2689,"min":2687,"totalFiles":8,"totalSize":21503}',
 'partitionsOptimized': 8,
 'numBatches': 8,
 'totalConsideredFiles': 8,
 'totalFilesSkipped': 0,
 'preserveInsertionOrder': False,
 'plannerStrategy': 'zOrder',
 'preservedStableOrder': False,
 'maxBinSpanFiles': 1}

## Verify Gold

In [5]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table())
print(gold_df)

# Slide-5 deliverable: "Gold p50/p95/cost qua ≥ 7 ngày". Make that explicit.
n_dates = gold_df.select("date").n_unique()
n_models = gold_df.select("model").n_unique()
print(
    f"\n──── Gold deliverable metrics ────\n"
    f"  Distinct dates:   {n_dates:>3}   (target ≥ 7)\n"
    f"  Distinct models:  {n_models:>3}\n"
    f"  Total Gold rows:  {gold_df.height:>3}   (= dates × models)"
)
assert n_dates >= 7, (
    f"Gold has only {n_dates} dates — slide deliverable requires ≥ 7. "
    "Re-run `make data` (the generator spreads across 7 UTC days)."
)

shape: (24, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ model      ┆ p50_latenc ┆ p95_laten ┆ total_pro ┆ total_com ┆ error_rat ┆ cost_usd  │
│ ---        ┆ ---        ┆ y_ms       ┆ cy_ms     ┆ mpt_token ┆ pletion_t ┆ e         ┆ ---       │
│ date       ┆ str        ┆ ---        ┆ ---       ┆ s         ┆ okens     ┆ ---       ┆ f64       │
│            ┆            ┆ f64        ┆ f64       ┆ ---       ┆ ---       ┆ f64       ┆           │
│            ┆            ┆            ┆           ┆ decimal[3 ┆ decimal[3 ┆           ┆           │
│            ┆            ┆            ┆           ┆ 8,0]      ┆ 8,0]      ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-04-02 ┆ claude-hai ┆ 565.0      ┆ 1140.0    ┆ 16651653  ┆ 8245961   ┆ 0.049327  ┆ 46.305166 │
│            ┆ ku-4-5     ┆            ┆           ┆           ┆           ┆

## Gold — kiểm tra đầy đủ tiêu chí (thêm bởi học viên)

Notebook gốc chưa assert toàn bộ điều kiện Gold, nên cell này tự kiểm tra từng điều kiện của rubric.

In [6]:
gold_df = pl.from_arrow(DeltaTable(GOLD).to_pyarrow_table()).sort(["date", "model"])
with pl.Config(tbl_rows=40, tbl_width_chars=160, fmt_str_lengths=20):
    print(gold_df.select("date", "model", "p50_latency_ms", "p95_latency_ms",
                         "error_rate", pl.col("cost_usd").round(2)))

rows_per_date = gold_df.group_by("date").agg(pl.len().alias("n_rows"),
                                             pl.col("model").n_unique().alias("n_models")).sort("date")
print(rows_per_date)

gold_checks = {
    "dates >= 7": gold_df["date"].n_unique() >= 7,
    "models == 3": gold_df["model"].n_unique() == 3,
    "every date has 3 models": bool((rows_per_date["n_models"] == 3).all()),
    "p50 <= p95 on every row": bool((gold_df["p50_latency_ms"] <= gold_df["p95_latency_ms"]).all()),
    "cost_usd > 0 on every row": bool((gold_df["cost_usd"] > 0).all()),
    "error_rate in [0, 1]": bool(gold_df["error_rate"].is_between(0, 1).all()),
}
for k, v in gold_checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(gold_checks.values())

print("\nStorage layer:")
for name, p in [("Bronze", BRONZE), ("Silver", SILVER), ("Gold", GOLD)]:
    print(f"  {name:6s} {p}  versions={DeltaTable(p).version() + 1}  rows={DeltaTable(p).to_pyarrow_table().num_rows:,}")

shape: (24, 6)
┌────────────┬───────────────────┬────────────────┬────────────────┬────────────┬──────────┐
│ date       ┆ model             ┆ p50_latency_ms ┆ p95_latency_ms ┆ error_rate ┆ cost_usd │
│ ---        ┆ ---               ┆ ---            ┆ ---            ┆ ---        ┆ ---      │
│ date       ┆ str               ┆ f64            ┆ f64            ┆ f64        ┆ f64      │
╞════════════╪═══════════════════╪════════════════╪════════════════╪════════════╪══════════╡
│ 2026-04-01 ┆ claude-haiku-4-5  ┆ 567.0          ┆ 1121.0         ┆ 0.04936    ┆ 33.18    │
│ 2026-04-01 ┆ claude-opus-4-7   ┆ 3045.0         ┆ 6035.65        ┆ 0.049735   ┆ 200.25   │
│ 2026-04-01 ┆ claude-sonnet-4-6 ┆ 1381.0         ┆ 2752.0         ┆ 0.050408   ┆ 244.15   │
│ 2026-04-02 ┆ claude-haiku-4-5  ┆ 565.0          ┆ 1140.0         ┆ 0.049327   ┆ 46.31    │
│ 2026-04-02 ┆ claude-opus-4-7   ┆ 3004.0         ┆ 5972.0         ┆ 0.052803   ┆ 291.29   │
│ 2026-04-02 ┆ claude-sonnet-4-6 ┆ 1388.0         ┆ 274

  Silver C:\CODE\AITHUCCHIEN\LABS\K4-Track02-Day18-LamHoangPhuc-2A202602582-Lakehouse-Lab\_lakehouse\silver\llm_calls  versions=1  rows=190,052
  Gold   C:\CODE\AITHUCCHIEN\LABS\K4-Track02-Day18-LamHoangPhuc-2A202602582-Lakehouse-Lab\_lakehouse\gold\llm_daily_metrics  versions=2  rows=24


## ✅ Deliverable check
- [ ] All three tables exist under `_lakehouse/{bronze,silver,gold}/`
- [ ] Silver has fewer rows than Bronze (dedup worked)
- [ ] Gold spans ≥ 7 dates × 3 models (slide §8 medallion contract)
- [ ] Cost & error_rate columns populated and non-zero

## 📝 Giải thích kết quả (học viên)
**Số đo:** Bronze 200,000 dòng → Silver 190,052 dòng (dedup loại **9,948** bản trùng `request_id`). Gold 24 dòng = **8 ngày × 3 model**;
mọi điều kiện kiểm tra ở cell bổ sung đều PASS: p50 ≤ p95, `cost_usd > 0`, `error_rate ∈ [0,1]` (thực tế ≈ 0.042–0.062). Ba bảng nằm tại `_lakehouse/bronze|silver|gold/…`.

**Đọc số:** p50 latency Haiku ≈ 560 ms, Sonnet ≈ 1,380 ms, Opus ≈ 3,000 ms; p95 ≈ 2× p50 cho cả 3 model. Một ngày đầy đủ Sonnet tốn ≈ $345 so với Opus ≈ $290
(Sonnet nhiều request hơn hẳn — 33M prompt tokens/ngày vs 5.5M của Opus) và Haiku ≈ $46. Giá là giá minh họa của lab.

**Dedup ở Silver giải quyết gì?** Bronze cố ý chứa retry (cùng `request_id` ghi lại nhiều lần — mô phỏng client/at-least-once delivery).
Nếu không dedup, Gold sẽ đếm trùng token → cost bị thổi phồng và error rate lệch. Silver dùng `ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY ts)` giữ bản đầu tiên, đồng thời parse JSON thành cột có kiểu.

**Vì sao dashboard đọc Gold?** Gold là bảng nhỏ (24 dòng), đã tổng hợp theo đúng câu hỏi nghiệp vụ (date × model) → dashboard nhanh, rẻ và mọi người dùng chung một định nghĩa metric, thay vì mỗi dashboard tự quét 190K dòng Silver.

**Phát hiện khi đối chiếu:** dữ liệu sinh ra phủ 7 ngày **UTC**, nhưng Gold có 8 ngày vì `CAST(ts AS DATE)` trong DuckDB dùng múi giờ phiên làm việc (máy tôi: Asia/Ho_Chi_Minh, UTC+7).
Hệ quả: 2026-04-01 chỉ có 17 giờ (cost Haiku $33.18 ≈ 17/24 × $46) và 2026-04-08 chỉ có 7 giờ ($13.50 ≈ 7/24 × $46). Ở production nên cố định `SET TimeZone='UTC'` hoặc chọn rõ múi giờ báo cáo,
nếu không cùng một pipeline chạy ở hai máy khác múi giờ sẽ ra hai bảng Gold khác nhau. Cách tính `error_rate = AVG(status <> 'ok')` và cost = tokens × giá/1M là phù hợp với Silver đã dedup.
